# ESM-DMS real-data analysis

This notebook runs the real-data ESM-DMS workflow for cellular datasets `TpoR`, `Ube4b`, and `BRCA1`, and viral datasets `BF520` and `BG505`.

It intentionally avoids the outdated `esmdmsfunctions.py`, `mega_analysis.py`, and `analysis_helpers.py` modules. Raw data parsing and current ESM-DMS objects come from `esmDMS.py`; the small cache-loading and summary helpers below are local to this notebook.

Outputs are written under `data/esm_data_analysis/`.


In [ ]:
from pathlib import Path
import itertools
import pickle
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break
    else:
        raise FileNotFoundError("Could not find esmDMS.py from the current working directory.")

DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
SEQUENCE_DIR = DATA_DIR / "sequence_data"
ANALYSIS_DIR = DATA_DIR / "esm_data_analysis"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
print(REPO_ROOT)


In [ ]:
from esmDMS import CellularDMSInput, ViralDMSInput, ESMDMSConfig, esmDMS
from popDMS import InferenceResult, mini_infer_esm


## Dataset Registry

Cellular datasets use MaveDB nucleotide-count files. Viral datasets use paired mutant-DNA and mutant-virus codon-count files.


In [ ]:
CELLULAR_DATASETS = {
    "TpoR": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    ),
    "Ube4b": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "Ube4b_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "Ube4b_nucleotide_counts.csv",
    ),
    "BRCA1": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "BRCA1_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "BRCA1_nucleotide_counts.csv",
    ),
}

VIRAL_DATASETS = {
    "BF520": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BF520_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BF520_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
    "BG505": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BG505_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BG505_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
}

DATASETS = {**CELLULAR_DATASETS, **VIRAL_DATASETS}
DATASET_KIND = {
    **{name: "cellular" for name in CELLULAR_DATASETS},
    **{name: "viral" for name in VIRAL_DATASETS},
}

pd.DataFrame(
    {"dataset": name, "kind": DATASET_KIND[name], "embedding_path": str(SEQUENCE_DIR / name)}
    for name in DATASETS
)


## Controls

`LAYERS = None` uses every cached layer for each dataset. Set it to a short list, for example `[0, 16, 33]`, while iterating. `FORCE_RECOMPUTE = False` loads cached inference results when compatible caches exist.


In [ ]:
LAYERS = None
NORMALIZE = "none"  # current options: "none", "cross_feature", "per_feature"
REPLICATES = None   # example: [1, 2, 3]
FORCE_RECOMPUTE = False
CALC_ERROR_BARS = False
EVERY_N_PLOT = 6

BASE_CONFIG = ESMDMSConfig(
    embedding_model="esm2_t33_650M_UR50D",
    embedding_method="per_residue",
    local_or_disk="disk",
    save_dir=str(SEQUENCE_DIR),
)


## Local Helpers

These helpers replace the old analysis modules. They understand the compact cached layout in `data/sequence_data/{dataset}`: shared `inference_metadata.pkl` plus per-layer `layer{i}_seq_to_emb.pkl` files.


In [ ]:
# TODO(esmDMS.py): expose compact cache discovery/loading for existing files named
# layer{i}_seq_to_emb.pkl, inference_metadata.pkl, and inference_results*.pkl.
# The current esmDMS cache path methods use newer names, so this notebook keeps a
# local compatibility layer rather than importing deprecated helper modules.

NORMALIZE_CACHE_NAMES = {
    "none": "inference_results.pkl",
    "cross_feature": "inference_results_cross_feature.pkl",
    "per_feature": "inference_results_per_feature.pkl",
}
LEGACY_NORMALIZE_CACHE_NAMES = {
    "cross_feature": "inference_results_by_layer.pkl",
    "per_feature": "inference_results_by_layer_dim.pkl",
}


def detect_layers(dataset_dir):
    dataset_dir = Path(dataset_dir)
    layers = []
    for path in dataset_dir.glob("layer*_seq_to_emb.pkl"):
        match = re.fullmatch(r"layer(\d+)_seq_to_emb\.pkl", path.name)
        if match:
            layers.append(int(match.group(1)))
    return sorted(layers)


def cache_path_for(dataset_dir, normalize=NORMALIZE):
    dataset_dir = Path(dataset_dir)
    cache_name = NORMALIZE_CACHE_NAMES.get(normalize)
    if cache_name is None:
        raise ValueError(f"Unsupported normalization scheme: {normalize!r}")
    return dataset_dir / cache_name


def cache_candidates(dataset_dir, normalize=NORMALIZE):
    dataset_dir = Path(dataset_dir)
    candidates = [cache_path_for(dataset_dir, normalize)]
    legacy_name = LEGACY_NORMALIZE_CACHE_NAMES.get(normalize)
    if legacy_name is not None:
        candidates.append(dataset_dir / legacy_name)
    return candidates


def find_cached_result_path(dataset_dir, normalize=NORMALIZE):
    for path in cache_candidates(dataset_dir, normalize):
        if path.exists():
            return path
    return None


def load_cached_result(dataset_dir, normalize=NORMALIZE):
    path = find_cached_result_path(dataset_dir, normalize)
    if path is None:
        return None, None
    with path.open("rb") as handle:
        return path, pickle.load(handle)


def result_parts(result):
    if isinstance(result, InferenceResult):
        return result.s, result.s_joint, result.error_bars, result.s_joint_error_bars, result.icov, result.gamma_opt
    if isinstance(result, (list, tuple)) and len(result) == 6:
        return result
    raise TypeError(f"Unsupported inference result payload: {type(result)!r}")


def normalize_feature_dict(seq_to_feature, normalize=NORMALIZE):
    if normalize == "none":
        return seq_to_feature
    seq_ids = list(seq_to_feature)
    features = np.asarray([seq_to_feature[seq_id] for seq_id in seq_ids])
    features = esmDMS._normalize_features(features, normalize)
    return dict(zip(seq_ids, features))


def load_sequence_dataframe(dataset_dir, replicates=REPLICATES):
    metadata_path = Path(dataset_dir) / "inference_metadata.pkl"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing compact metadata file: {metadata_path}")
    df = pd.read_pickle(metadata_path).copy()
    if "seq_id" in df.columns and "SequenceIndex" not in df.columns:
        df = df.rename(columns={"seq_id": "SequenceIndex"})
    if replicates is not None:
        df = df[df["Replicate"].isin(replicates)].copy()
    return df


def load_layer_features(dataset_dir, layer, normalize=NORMALIZE):
    path = Path(dataset_dir) / f"layer{layer}_seq_to_emb.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Missing compact embedding file: {path}")
    with path.open("rb") as handle:
        payload = pickle.load(handle)
    if isinstance(payload, dict):
        seq_to_feature = payload
    else:
        arr = np.asarray(payload)
        seq_to_feature = {idx: arr[idx] for idx in range(arr.shape[0])}
    return normalize_feature_dict(seq_to_feature, normalize)


def run_or_load_inference(dataset, layers=None, normalize=NORMALIZE, replicates=REPLICATES, force_recompute=FORCE_RECOMPUTE):
    dataset_dir = SEQUENCE_DIR / dataset
    selected_layers = detect_layers(dataset_dir) if layers is None else list(layers)
    if not selected_layers:
        raise FileNotFoundError(f"No layer*_seq_to_emb.pkl files found in {dataset_dir}")

    cache_path, cached = load_cached_result(dataset_dir, normalize)
    if cached is not None and not force_recompute:
        processed = cached[2]
        missing = [layer for layer in selected_layers if layer not in processed]
        if not missing:
            print(f"{dataset}: loaded {len(selected_layers)} layers from {cache_path.name}")
            return (cached[0], cached[1], {layer: processed[layer] for layer in selected_layers}, cached[3], cached[4])
        print(f"{dataset}: cache {cache_path.name} is missing layers {missing}; recomputing selected layers")

    sequence_dataframe = load_sequence_dataframe(dataset_dir, replicates=replicates)
    processed = {}
    for layer in selected_layers:
        print(f"{dataset}: running inference for layer {layer}")
        seq_to_feature = load_layer_features(dataset_dir, layer, normalize=normalize)
        result = mini_infer_esm(
            sequence_dataframe,
            seq_to_feature,
            gamma=None,
            corr_cutoff_pct=0.5,
            max_reads=1e2,
            plot_gamma=False,
            verbose=False,
            calc_error_bars=CALC_ERROR_BARS,
        )
        processed[layer] = list(result_parts(result))

    result_tuple = (None, None, processed, None, None)
    cache_path = cache_path_for(dataset_dir, normalize)
    with cache_path.open("wb") as handle:
        pickle.dump(result_tuple, handle, protocol=4)
    print(f"{dataset}: saved recomputed results to {cache_path}")
    return result_tuple


def safe_corr(x, y, corr_fn):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return corr_fn(x, y).statistic


## Process Raw Data

This confirms the real input files can be parsed by the current `esmDMS.py` data classes and processing methods. The later inference section uses cached compact embeddings by default, avoiding ESM re-embedding.


In [ ]:
processed = {}
processing_rows = []

for dataset, input_data in DATASETS.items():
    cfg = ESMDMSConfig(
        embedding_model=BASE_CONFIG.embedding_model,
        embedding_method=BASE_CONFIG.embedding_method,
        local_or_disk=BASE_CONFIG.local_or_disk,
        save_dir=str(SEQUENCE_DIR / dataset),
        dataset_name=dataset,
    )
    runner = esmDMS(input_data=input_data, config=cfg)
    runner.process_raw_data(drop_stop_codons=True)
    processed[dataset] = runner

    df = runner.sequence_dataframe
    processing_rows.append({
        "dataset": dataset,
        "kind": DATASET_KIND[dataset],
        "rows": len(df),
        "sequence_count": df["SequenceIndex"].nunique(),
        "replicate_count": df["Replicate"].nunique(),
        "generation_count": df["Generation"].nunique(),
    })

processing_summary = pd.DataFrame(processing_rows)
processing_summary.to_csv(TABLE_DIR / "raw_processing_summary.csv", index=False)
processing_summary


## Validate Cached Embeddings

In [ ]:
cache_rows = []

for dataset in DATASETS:
    path = SEQUENCE_DIR / dataset
    detected_layers = detect_layers(path)
    raw_embedding_file = path / f"{dataset}_embeddings.pkl"
    metadata_file = path / "inference_metadata.pkl"
    seq_map_file = path / "seq_id_map.pkl"
    cache_file = find_cached_result_path(path, NORMALIZE)

    cache_rows.append({
        "dataset": dataset,
        "embedding_path": str(path),
        "raw_embeddings": raw_embedding_file.exists(),
        "metadata": metadata_file.exists(),
        "seq_id_map": seq_map_file.exists(),
        "cached_inference": cache_file.name if cache_file else None,
        "n_layers_detected": len(detected_layers),
        "first_layer": min(detected_layers) if detected_layers else np.nan,
        "last_layer": max(detected_layers) if detected_layers else np.nan,
    })

cache_summary = pd.DataFrame(cache_rows)
cache_summary.to_csv(TABLE_DIR / "embedding_cache_summary.csv", index=False)
cache_summary


## Run Or Load ESM-DMS Inference

By default this loads cached `inference_results*.pkl` files. If a cache is absent, missing a selected layer, or `FORCE_RECOMPUTE` is `True`, the notebook rebuilds inference from `inference_metadata.pkl` and `layer{i}_seq_to_emb.pkl` using `popDMS.mini_infer_esm`.


In [ ]:
all_results = {}

for dataset in DATASETS:
    dataset_layers = detect_layers(SEQUENCE_DIR / dataset) if LAYERS is None else LAYERS
    print(f"{dataset}: {len(dataset_layers)} selected layers")
    all_results[dataset] = run_or_load_inference(
        dataset,
        layers=dataset_layers,
        normalize=NORMALIZE,
        replicates=REPLICATES,
        force_recompute=FORCE_RECOMPUTE,
    )


## Inference Result Summary

In [ ]:
inference_rows = []

for dataset, results in all_results.items():
    detailed = results[2]
    for layer, layer_result in sorted(detailed.items()):
        s, s_joint, error_bars, s_joint_error_bars, icov, gamma_opt = result_parts(layer_result)
        inference_rows.append({
            "dataset": dataset,
            "kind": DATASET_KIND[dataset],
            "layer": layer,
            "n_replicates": s.shape[0],
            "n_dimensions": s.shape[1],
            "gamma_opt": gamma_opt,
            "s_joint_mean": float(np.nanmean(s_joint)),
            "s_joint_std": float(np.nanstd(s_joint)),
        })

inference_summary = pd.DataFrame(inference_rows)
inference_summary.to_csv(TABLE_DIR / "inference_result_summary.csv", index=False)
inference_summary.head()


## Replicate Consistency Summaries

In [ ]:
consistency_rows = []

for dataset, results in all_results.items():
    for layer, layer_result in sorted(results[2].items()):
        s = result_parts(layer_result)[0]
        seq_to_feature = load_layer_features(SEQUENCE_DIR / dataset, layer, normalize=NORMALIZE)
        seq_ids = list(seq_to_feature)
        feature_matrix = np.asarray([seq_to_feature[seq_id] for seq_id in seq_ids])
        rep_fitness = np.asarray([feature_matrix @ s_rep for s_rep in s])
        for i, j in itertools.combinations(range(s.shape[0]), 2):
            consistency_rows.append({
                "dataset": dataset,
                "kind": DATASET_KIND[dataset],
                "layer": layer,
                "replicate_i": i + 1,
                "replicate_j": j + 1,
                "selection_pearson": safe_corr(s[i], s[j], pearsonr),
                "selection_spearman": safe_corr(s[i], s[j], spearmanr),
                "fitness_pearson": safe_corr(rep_fitness[i], rep_fitness[j], pearsonr),
                "fitness_spearman": safe_corr(rep_fitness[i], rep_fitness[j], spearmanr),
            })

consistency = pd.DataFrame(consistency_rows)
consistency.to_csv(TABLE_DIR / "replicate_pair_consistency.csv", index=False)

consistency_summary = (
    consistency
    .groupby(["dataset", "kind", "layer"], as_index=False)
    [["selection_pearson", "selection_spearman", "fitness_pearson", "fitness_spearman"]]
    .mean()
)
consistency_summary.to_csv(TABLE_DIR / "replicate_consistency_summary.csv", index=False)
consistency_summary.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

sns.lineplot(
    data=consistency_summary,
    x="layer",
    y="selection_pearson",
    hue="dataset",
    style="kind",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Selection coefficient replicate consistency")
axes[0].set_ylabel("Mean pairwise Pearson r")

sns.lineplot(
    data=consistency_summary,
    x="layer",
    y="fitness_pearson",
    hue="dataset",
    style="kind",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Inferred fitness replicate consistency")
axes[1].set_ylabel("Mean pairwise Pearson r")

for ax in axes:
    ax.set_ylim(-0.1, 1.05)
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "replicate_consistency_by_layer.png", dpi=150, bbox_inches="tight")
plt.show()


## Per-Dataset Cross-Replicate Plots

This local plotting block replaces the old shared plotting helper. To keep notebooks responsive, it plots every `EVERY_N_PLOT` layer by default.


In [ ]:
cross_rep_dir = FIGURE_DIR / "cross_replicate"
cross_rep_dir.mkdir(parents=True, exist_ok=True)

for dataset, results in all_results.items():
    dataset_dir = cross_rep_dir / dataset
    dataset_dir.mkdir(parents=True, exist_ok=True)
    layers = sorted(results[2])
    plot_layers = layers[::EVERY_N_PLOT] or layers

    for layer in plot_layers:
        s = result_parts(results[2][layer])[0]
        pairs = list(itertools.combinations(range(s.shape[0]), 2))
        if not pairs:
            continue
        n_cols = min(3, len(pairs))
        n_rows = int(np.ceil(len(pairs) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), squeeze=False)
        fig.suptitle(f"{dataset} layer {layer}: selection coefficients", fontsize=13)
        for ax_idx, (i, j) in enumerate(pairs):
            ax = axes[ax_idx // n_cols][ax_idx % n_cols]
            sns.scatterplot(x=s[i], y=s[j], s=14, alpha=0.55, edgecolor=None, ax=ax)
            r = safe_corr(s[i], s[j], pearsonr)
            rho = safe_corr(s[i], s[j], spearmanr)
            ax.set_title(f"Rep {i + 1} vs {j + 1}: r={r:.3f}, rho={rho:.3f}", fontsize=10)
            ax.set_xlabel(f"Rep {i + 1}")
            ax.set_ylabel(f"Rep {j + 1}")
        for ax_idx in range(len(pairs), n_rows * n_cols):
            axes[ax_idx // n_cols][ax_idx % n_cols].set_visible(False)
        fig.tight_layout()
        fig.savefig(dataset_dir / f"layer{layer}_selection_replicate_scatter.png", dpi=150, bbox_inches="tight")
        plt.close(fig)


## Optional Shuffled-Frequency Control

This control is not implemented in current `esmDMS.py` as a reusable API. The analysis is marked as a TODO rather than importing old helper modules.


In [ ]:
RUN_SHUFFLED_CONTROL = False

# TODO(esmDMS.py): add a first-class shuffled-frequency control that shuffles
# frequencies within each (Replicate, Generation), reruns inference, and returns
# the same result object used by run_feature_inference. Keeping that logic in the
# library will avoid reviving deprecated analysis helpers in notebooks.

if RUN_SHUFFLED_CONTROL:
    raise NotImplementedError(
        "Shuffled-frequency controls should be added to esmDMS.py before this notebook runs them."
    )
